In [1]:
import pandas as pd
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/features.csv")
X = df.drop(columns=['user_id', 'is_churn'])
y = df['is_churn']

In [3]:
# We MUST evaluate on a holdout set so the AI can't cheat
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with open("../outputs/xgboost_churn_model.pkl", "rb") as f:
    model = pickle.load(f)

print("Making predictions on the holdout test set...")
preds = model.predict(X_test)

Making predictions on the holdout test set...


In [4]:
#--- BUSINESS FINANCIAL ASSUMPTIONS ---
# KKBOX standard monthly plan is roughly 149 NTD (New Taiwan Dollars)
MONTHLY_FEE = 149  
DISCOUNT_COST = 30 # It costs 30 NTD to offer a "Stay with us" discount
SAVE_RATE = 0.50   # Assume the discount successfully saves 50% of the churners

print("Running financial simulation...")
baseline_revenue = 0
ai_revenue = 0

# Calculate outcome for every single user in the test set
for actual, predicted in zip(y_test, preds):
    # 1. BASELINE (No AI, we do nothing)
    if actual == 0:
        baseline_revenue += MONTHLY_FEE # Kept the user
        
    # 2. AI STRATEGY
    if predicted == 1:
        # AI flags them. We spend $30 on a discount.
        if actual == 1:
            # TRUE POSITIVE: They were going to leave. We have a 50% chance to save them.
            ai_revenue += (MONTHLY_FEE - DISCOUNT_COST) * SAVE_RATE
        else:
            # FALSE POSITIVE: They were happy. We just gave away $30 for no reason.
            ai_revenue += (MONTHLY_FEE - DISCOUNT_COST)
    else:
        # AI says they are happy. We do nothing.
        if actual == 0:
            # TRUE NEGATIVE: They stayed, we get full revenue.
            ai_revenue += MONTHLY_FEE
        # (If False Negative, they leave and we get $0, which is mathematically the same as doing nothing)

Running financial simulation...


In [6]:
# Extrapolate the 20% test set to the entire 860k user base
total_baseline = baseline_revenue * 5
total_ai = ai_revenue * 5
financial_lift = total_ai - total_baseline

print(f"\n{'='*45}")
print(f"THE EXECUTIVE SUMMARY")
print(f"{'='*45}")
print(f"Projected Monthly Revenue without AI: ${total_baseline:,.2f}")
print(f"Projected Monthly Revenue with AI:    ${total_ai:,.2f}")
print(f"---------------------------------------------")
print(f"NET MONTHLY VALUE GENERATED:          +${financial_lift:,.2f}")
print(f"{'='*45}")


THE EXECUTIVE SUMMARY
Projected Monthly Revenue without AI: $116,148,480.00
Projected Monthly Revenue with AI:    $118,421,612.50
---------------------------------------------
NET MONTHLY VALUE GENERATED:          +$2,273,132.50
